In [4]:
!pip install easyocr transformers openai-whisper gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 30.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 12.6 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=a219aa5665cd394eb5b19b2879f63eb1e2dd56d62b49760293bc8ab816e05d1d
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [ ]:
!pip install easyocr openai-whisper
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import easyocr
import whisper
import gradio as gr

# 1. Initialize Models (This will download them the first time)
# OCR for Image-to-Text
reader = easyocr.Reader(['en'])

# Translation (English to Urdu)
# We use a Transformer model for contextual accuracy
model_name = "Helsinki-NLP/opus-mt-en-ur"
tokenizer = AutoTokenizer.from_pretrained(model_name)
translation_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Voice (Transcription and Translation)
voice_model = whisper.load_model("base")

# 2. Define Logic Functions
def translate_text(text):
    if not text.strip(): return ""
    inputs = tokenizer(text, return_tensors="pt")
    # Move model to CPU if GPU is not available
    if not torch.cuda.is_available():
        translation_model.to('cpu')
        inputs = {k: v.to('cpu') for k, v in inputs.items()}

    translated_tokens = translation_model.generate(**inputs, max_length=512)
    translated_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
    return translated_text

def translate_image(img):
    # OCR to get text
    results = reader.readtext(img)
    extracted_text = " ".join([res[1] for res in results])
    # Translate extracted text
    return translate_text(extracted_text)

def translate_voice(audio_path):
    # Whisper transcribes and can translate to English,
    # but for Urdu we transcribe first then translate
    result = voice_model.transcribe(audio_path)
    return translate_text(result['text'])

# 3. Create the App Interface (using Gradio)
with gr.Blocks() as demo:
    gr.Markdown("# 🇵🇰 AI English to Urdu Translator")

    with gr.Tab("Text Translation"):
        txt_input = gr.Textbox(label="Enter English")
        txt_output = gr.Textbox(label="Urdu Result")
        btn_txt = gr.Button("Translate Text")
        btn_txt.click(translate_text, inputs=txt_input, outputs=txt_output)

    with gr.Tab("Image Translation"):
        img_input = gr.Image(type="filepath", label="Upload Image/Screenshot")
        img_output = gr.Textbox(label="Urdu Result")
        btn_img = gr.Button("Extract & Translate")
        btn_img.click(translate_image, inputs=img_input, outputs=img_output)

    with gr.Tab("Voice Translation"):
        aud_input = gr.Audio(type="filepath", label="Record or Upload Audio")
        aud_output = gr.Textbox(label="Urdu Result")
        btn_aud = gr.Button("Transcribe & Translate")
        btn_aud.click(translate_voice, inputs=aud_input, outputs=aud_output)

demo.launch(debug=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 30.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.6 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=b2e4011f711150a4710c4973b6443a9c47acc6245b3b91bd835f463fc90bad96
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/306M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]


  0%|                                               | 0.00/139M [00:00<?, ?iB/s]
  8%|███▏                                   | 11.2M/139M [00:00<00:01, 115MiB/s]
 16%|██████                                | 22.1M/139M [00:00<00:01, 94.1MiB/s]
 23%|████████▌                             | 31.3M/139M [00:00<00:01, 89.6MiB/s]
 29%|███████████                           | 40.3M/139M [00:00<00:01, 91.3MiB/s]
 35%|█████████████▍                        | 49.1M/139M [00:00<00:01, 68.5MiB/s]
 41%|███████████████▍                      | 56.3M/139M [00:00<00:01, 58.9MiB/s]
 45%|█████████████████▏                    | 62.9M/139M [00:00<00:01, 61.2MiB/s]
 50%|██████████████████▉                   | 69.1M/139M [00:01<00:01, 57.4MiB/s]
 54%|████████████████████▌                 | 74.9M/139M [00:01<00:01, 53.4MiB/s]
 58%|██████████████████████                | 80.2M/139M [00:01<00:01, 51.2MiB/s]
 62%|███████████████████████▍              | 85.5M/139M [00:01<00:01, 52.2MiB/s]
 65%|██████████████████████

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://abd8c827beebd2eab9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
